# Normalization XB47Y
y la felicidad que? - Canserbero

In [1]:
%load_ext autoreload
%autoreload 2
import tools_EEG as TEEG

## 1. Z-score per recording
the goal is to calculate the z-score per npz/mat file. For this we calculate the mean and std dev. of all the files and then we applied that for every recording
Store all the data in a json? easy access idea. 

## 1.1 Testing the code before building the function
Check keys from the npz

### Step 1: key inspection

In [2]:
from pathlib import Path
import numpy as np

# Cambia esto por la ruta a uno de tus archivos reales
ruta_archivo = Path("/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_testALL_25032026/XB47Y_182_preproc_full.npz")

# Cargar el archivo
data = np.load(ruta_archivo, allow_pickle=True)

# Ver qué claves tiene
print("Claves encontradas:", list(data.keys()))
# expected: Claves encontradas: ['X', 'mu', 'sigma', 'fs', 'channel_names', 'source_file', 'seizure_onsets', 'T0', 'TF']

Claves encontradas: ['X', 'mu', 'sigma', 'fs', 'channel_names', 'source_file', 'seizure_onsets', 'T0', 'TF']


### Step 2: array inspection
Read the array X, and check if there is anything strange in it. 

In [3]:
# Leer el array X
X = data["X"]

print("Type of data:", X.dtype)
print("(shape):", X.shape)
print(f"  → Channels (C): {X.shape[0]}")
print(f"  →  Samples (N): {X.shape[1]}")
print()
print("Val mínimo:", np.nanmin(X))
print("Val máximo:", np.nanmax(X))
print("¿NaN?:  ", np.any(np.isnan(X)))
print("¿Inf?:  ", np.any(np.isinf(X)))



#Tipo de dato: float64
#Forma (shape): (19, 153600)
#  → Canales (C): 19
#  → Muestras (N): 153600

#Valor mínimo: -245.3
#Valor máximo:  312.7
#¿Hay NaN?:   False
#¿Hay Inf?:   False

Type of data: float32
(shape): (2, 3018060)
  → Channels (C): 2
  →  Samples (N): 3018060

Val mínimo: -376.70462
Val máximo: 329.29227
¿NaN?:   False
¿Inf?:   False


### Step 3: file inspection
** Now lets work with the whole directory **
Lets check the number of files

In [4]:
from pathlib import Path

# Change this to your actual input folder
input_dir = Path("/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_testALL_25032026/")

# Find all .npz files
all_files = sorted(input_dir.glob("*.npz"))

print(f"Files found: {len(all_files)}")
#for f in all_files:
    #print(" ", f.name)
# expected 
# Files found: 5
#  patient01.npz
#  patient02.npz
#  patient03.npz


Files found: 183


### Step 4: calculate mean and std per channel
Chan's parallel update — numerically stable, no need to load everything into memory at once

In [5]:
# Number of channels (we know it's 2, but we read it dynamically)
N_CHANNELS = np.load(all_files[0], allow_pickle=True)["X"].shape[0]
print(f"Number of channels: {N_CHANNELS}")

# One accumulator per channel
ch_count = np.zeros(N_CHANNELS, dtype=np.float64)
ch_mean  = np.zeros(N_CHANNELS, dtype=np.float64)
ch_M2    = np.zeros(N_CHANNELS, dtype=np.float64)

skipped = []

for path in all_files:
    try:
        data = np.load(path, allow_pickle=True)
    except Exception as exc:
        print(f"[WARNING] Could not load '{path.name}': {exc}")
        skipped.append(path.name)
        continue

    if "X" not in data:
        print(f"[WARNING] '{path.name}' has no key 'X' — skipping.")
        skipped.append(path.name)
        continue

    X = data["X"]  # shape (C, N)

    for c in range(N_CHANNELS):
        row = X[c, :]
        finite_vals = row[np.isfinite(row)]

        if finite_vals.size == 0:
            continue

        b_count = finite_vals.size
        b_mean  = float(np.mean(finite_vals))
        b_M2    = float(np.var(finite_vals, ddof=0)) * b_count

        # Chan's parallel update
        combined_count = ch_count[c] + b_count
        delta          = b_mean - ch_mean[c]

        ch_mean[c]  = (ch_count[c] * ch_mean[c] + b_count * b_mean) / combined_count
        ch_M2[c]   += b_M2 + delta**2 * (ch_count[c] * b_count) / combined_count
        ch_count[c] = combined_count

    #print(f"  Processed '{path.name}'")

# Final per-channel scalars
ch_std = np.sqrt(ch_M2 / ch_count)

print("\n" + "=" * 45)
for c in range(N_CHANNELS):
    print(f"  Channel {c}  |  mean: {ch_mean[c]:.6f}  |  std: {ch_std[c]:.6f}  |  samples: {int(ch_count[c]):,}")
print("=" * 45)
print(f"Files skipped: {len(skipped)}")

#Expected output aprox :

#Channel 0  |  mean:  0.000032  |  std: 21.340000  |  samples: 602,754,813
#Channel 1  |  mean:  0.000071  |  std: 24.210000  |  samples: 602,754,813

Number of channels: 2

  Channel 0  |  mean: 0.000058  |  std: 27.632445  |  samples: 602,754,813
  Channel 1  |  mean: 0.000041  |  std: 16.593436  |  samples: 602,754,813
Files skipped: 0


### Step 5: data normalization per recording + save it in a new npz
Calculate aprox. stats parameters to better understand the global data

In [6]:
from pathlib import Path
import numpy as np

output_dir = Path("/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_28032026Normalized/")
output_dir.mkdir(parents=True, exist_ok=True)

METADATA_KEYS = (
    "mu", "sigma", "fs", "channel_names",
    "source_file", "seizure_onsets", "T0", "TF"
)

EPS = 1e-8
n_ok = 0
n_skipped = 0

# Opcional: guardar stats por recording en listas simples
recording_names = []
recording_shapes = []
recording_ok = []

for path in all_files:
    try:
        data = np.load(path, allow_pickle=True)
        X = data["X"]   # shape: (C, N)

        if X.shape[0] != N_CHANNELS:
            raise ValueError(
                f"Expected {N_CHANNELS} channels, got {X.shape[0]}"
            )

        # Normalización global por canal
        X_norm = np.empty_like(X, dtype=np.float32)
        for c in range(N_CHANNELS):
            X_norm[c, :] = (X[c, :] - ch_mean[c]) / (ch_std[c] + EPS)

        # Preservar metadata original
        metadata = {key: data[key] for key in METADATA_KEYS if key in data}

        # Guardar nuevo NPZ normalizado
        # Además guardamos la mean/std global usada para normalizar
        np.savez_compressed(
            output_dir / path.name,
            X=X_norm,
            global_mu=ch_mean.astype(np.float32),
            global_sigma=ch_std.astype(np.float32),
            normalization_type="global_channel_zscore",
            eps=np.float32(EPS),
            **metadata
        )

        recording_names.append(path.name)
        recording_shapes.append(X.shape)
        recording_ok.append(True)

        n_ok += 1
        #print(f"  ✓ '{path.name}'")

    except Exception as exc:
        print(f"[WARNING] Failed on '{path.name}': {exc}")
        recording_names.append(path.name)
        recording_shapes.append((-1, -1))
        recording_ok.append(False)
        n_skipped += 1

# Guardar stats globales en NPZ en vez de JSON
stats_npz_path = output_dir / "normalization_stats_global.npz"
np.savez_compressed(
    stats_npz_path,
    global_mu=ch_mean.astype(np.float32),
    global_sigma=ch_std.astype(np.float32),
    global_sample_count=ch_count.astype(np.int64),
    n_channels=np.int32(N_CHANNELS),
    eps=np.float32(EPS),
    recording_names=np.array(recording_names, dtype=object),
    recording_shapes=np.array(recording_shapes, dtype=object),
    recording_ok=np.array(recording_ok, dtype=bool),
)

print(f"\nDone. Saved: {n_ok}  |  Skipped: {n_skipped}")
print(f"Global stats written to: {stats_npz_path}")

  ✓ 'XB47Y_100_preproc_full.npz'
  ✓ 'XB47Y_101_preproc_full.npz'
  ✓ 'XB47Y_102_preproc_full.npz'
  ✓ 'XB47Y_103_preproc_full.npz'
  ✓ 'XB47Y_104_preproc_full.npz'
  ✓ 'XB47Y_105_preproc_full.npz'
  ✓ 'XB47Y_106_preproc_full.npz'
  ✓ 'XB47Y_107_preproc_full.npz'
  ✓ 'XB47Y_108_preproc_full.npz'
  ✓ 'XB47Y_109_preproc_full.npz'
  ✓ 'XB47Y_10_preproc_full.npz'
  ✓ 'XB47Y_110_preproc_full.npz'
  ✓ 'XB47Y_111_preproc_full.npz'
  ✓ 'XB47Y_112_preproc_full.npz'
  ✓ 'XB47Y_113_preproc_full.npz'
  ✓ 'XB47Y_114_preproc_full.npz'
  ✓ 'XB47Y_115_preproc_full.npz'
  ✓ 'XB47Y_116_preproc_full.npz'
  ✓ 'XB47Y_117_preproc_full.npz'
  ✓ 'XB47Y_118_preproc_full.npz'
  ✓ 'XB47Y_119_preproc_full.npz'
  ✓ 'XB47Y_11_preproc_full.npz'
  ✓ 'XB47Y_120_preproc_full.npz'
  ✓ 'XB47Y_121_preproc_full.npz'
  ✓ 'XB47Y_122_preproc_full.npz'
  ✓ 'XB47Y_123_preproc_full.npz'
  ✓ 'XB47Y_124_preproc_full.npz'
  ✓ 'XB47Y_125_preproc_full.npz'
  ✓ 'XB47Y_126_preproc_full.npz'
  ✓ 'XB47Y_127_preproc_full.npz'
  ✓ 'XB47Y_1

## Step 6: Sanity check function


* Verify that each normalized `.npz` file exists and matches the original file name
* Check that required keys are present (`X`, `global_mu`, `global_sigma`, `eps`)
* Ensure shapes are consistent between original and normalized data `(C, N)`
* Confirm all values are finite (no NaN or Inf) in both original and normalized signals
* Recompute Z-score using `global_mu` and `global_sigma` and compare with stored `X`
* Measure maximum absolute difference between recomputed and stored values
* Flag file as **pass** if values match within tolerance, otherwise **fail**
* Optionally compute per-channel mean and std for inspection (not expected to be 0/1 per file)
* At dataset level, verify overall mean ≈ 0 and std ≈ 1 across all normalized data


In [12]:
df_check = TEEG.sanity_check_global_zscore_npz_1_14(
    original_dir="/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_testALL_25032026/",
    normalized_dir="/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_28032026Normalized/"
)


=== Sanity Check Summary ===
status
pass    183
Name: count, dtype: int64


In [10]:
df_check.head()

,file,original_exists,normalized_exists,shape_original,shape_normalized,same_shape,has_required_keys,all_finite_original,all_finite_normalized,zscore_recomputed_match,max_abs_diff,mean_norm_ch0,std_norm_ch0,mean_norm_ch1,std_norm_ch1,status,reason
0,XB47Y_100_preproc_full.npz,True,True,"(2, 4472028)","(2, 4472028)",True,True,True,True,True,0.000002,-0.000004,0.839437,-0.000004,0.873959,pass,
1,XB47Y_101_preproc_full.npz,True,True,"(2, 2408445)","(2, 2408445)",True,True,True,True,True,0.000002,-0.000006,0.280173,0.000005,0.431584,pass,
2,XB47Y_102_preproc_full.npz,True,True,"(2, 4472028)","(2, 4472028)",True,True,True,True,True,0.000002,0.000064,1.312999,0.000099,1.342000,pass,
3,XB47Y_103_preproc_full.npz,True,True,"(2, 4472028)","(2, 4472028)",True,True,True,True,True,0.000002,-0.000016,1.264091,-0.000009,1.329985,pass,
4,XB47Y_104_preproc_full.npz,True,True,"(2, 4472028)","(2, 4472028)",True,True,True,True,True,0.000002,-0.000005,1.082205,-0.000005,1.089423,pass,
